# TCGA-LUAD: Somatic Mutation & Survival Analysis

This notebook analyzes somatic mutation data from the TCGA Lung Adenocarcinoma (LUAD) cohort to identify which driver gene mutations are associated with overall patient survival.

**Data source:** UCSC Xena — GDC TCGA Lung Adenocarcinoma (LUAD)  
**Files used:**
- `ESM.tsv.gz` — Ensemble somatic mutation calls (194,731 mutations across 575 patients)
- `phenotype.tsv` — Overall survival data (OS.time in days, OS event flag)

**Methods:** Mutation frequency analysis, binary mutation matrix construction, Kaplan-Meier survival estimation, log-rank hypothesis testing

In [ ]:
# Install survival analysis library if not already present
!pip install lifelines -q

In [ ]:
import gzip
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

## 1. Load Data

In [ ]:
with gzip.open('ESM.tsv.gz', 'rt') as f:
    mutations = pd.read_csv(f, sep='\t')

survival = pd.read_csv('phenotype.tsv', sep='\t')

print(f'Mutations: {mutations.shape[0]:,} rows, {mutations.shape[1]} columns')
print(f'Survival:  {survival.shape[0]} patients')
print(f'\nMutation columns: {mutations.columns.tolist()}')
print(f'Survival columns: {survival.columns.tolist()}')

## 2. Mutation Frequency — Top 20 Genes

In [ ]:
gene_counts = mutations['gene'].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
gene_counts.head(20).plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 20 Most Mutated Genes in TCGA-LUAD', fontsize=14, fontweight='bold')
ax.set_xlabel('Gene')
ax.set_ylabel('Mutation Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('mutation_frequency.png', dpi=150)
plt.show()

print('\nNote: TTN, MUC16, CSMD3 are large genes that accumulate mutations by size, not necessarily cancer drivers.')
print(f'TP53 rank: {list(gene_counts.index).index("TP53") + 1}')

## 3. Build Binary Mutation Matrix

For each patient, we mark whether they carry a mutation in any of the 8 known LUAD driver genes. These were selected based on their recurrence in LUAD literature (Cancer Genome Atlas Research Network, 2014).

In [ ]:
driver_genes = ['TP53', 'KRAS', 'EGFR', 'STK11', 'KEAP1', 'RBM10', 'BRAF', 'NF1']

mutation_matrix = pd.crosstab(mutations['sample'], mutations['gene'])[driver_genes]
mutation_matrix = (mutation_matrix > 0).astype(int)
mutation_matrix.reset_index(inplace=True)

print(f'Mutation matrix shape: {mutation_matrix.shape}')
print(f'\nMutation rates per gene:')
for gene in driver_genes:
    rate = mutation_matrix[gene].mean() * 100
    print(f'  {gene}: {rate:.1f}%')

## 4. Merge with Survival Data

Sample IDs contain tissue-type suffixes (e.g. `-01A`) that differ between files. We trim to the 12-character patient ID for consistent matching.

In [ ]:
mutation_matrix['patient'] = mutation_matrix['sample'].str[:12]
survival['patient'] = survival['sample'].str[:12]

merged = pd.merge(mutation_matrix, survival[['patient', 'OS.time', 'OS']], on='patient')

print(f'Merged dataset: {merged.shape[0]} patients with both mutation and survival data')
print(f'Median follow-up: {merged["OS.time"].median():.0f} days')
print(f'Events (deaths): {merged["OS"].sum()} ({merged["OS"].mean()*100:.1f}%)')

## 5. Kaplan-Meier Survival Analysis — All Driver Genes

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

pvalues = {}

for i, gene in enumerate(driver_genes):
    mutated = merged[merged[gene] == 1]
    not_mutated = merged[merged[gene] == 0]

    kmf = KaplanMeierFitter()
    kmf.fit(mutated['OS.time'], mutated['OS'], label=f'Mutated (n={len(mutated)})')
    kmf.plot_survival_function(ax=axes[i])

    kmf.fit(not_mutated['OS.time'], not_mutated['OS'], label=f'Not Mutated (n={len(not_mutated)})')
    kmf.plot_survival_function(ax=axes[i])

    result = logrank_test(mutated['OS.time'], not_mutated['OS.time'],
                          event_observed_A=mutated['OS'],
                          event_observed_B=not_mutated['OS'])
    pvalues[gene] = result.p_value

    sig = ' *' if result.p_value < 0.05 else ''
    axes[i].set_title(f'{gene} (p={result.p_value:.4f}){sig}', fontweight='bold')
    axes[i].set_xlabel('Days')
    axes[i].set_ylabel('Survival Probability')

plt.suptitle('Kaplan-Meier Survival Curves by Driver Gene Mutation — TCGA-LUAD', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('km_all_genes.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nLog-rank p-values:')
for gene, p in sorted(pvalues.items(), key=lambda x: x[1]):
    sig = ' <- significant' if p < 0.05 else ''
    print(f'  {gene}: {p:.4f}{sig}')

## 6. Key Finding — RBM10

RBM10 (RNA Binding Motif Protein 10) is an RNA splicing regulator. Loss-of-function mutations in RBM10 have been identified as a recurrent event in LUAD and may promote exon skipping in tumor suppressor transcripts. Here we show that RBM10-mutated patients have significantly worse overall survival (p=0.0228).

In [ ]:
gene = 'RBM10'
mutated = merged[merged[gene] == 1]
not_mutated = merged[merged[gene] == 0]

result = logrank_test(mutated['OS.time'], not_mutated['OS.time'],
                      event_observed_A=mutated['OS'],
                      event_observed_B=not_mutated['OS'])

kmf = KaplanMeierFitter()
fig, ax = plt.subplots(figsize=(10, 6))

kmf.fit(mutated['OS.time'], mutated['OS'], label=f'RBM10 Mutated (n={len(mutated)})')
kmf.plot_survival_function(ax=ax)

kmf.fit(not_mutated['OS.time'], not_mutated['OS'], label=f'RBM10 Not Mutated (n={len(not_mutated)})')
kmf.plot_survival_function(ax=ax)

ax.set_title(f'RBM10 Mutation vs Overall Survival in LUAD\nLog-rank p = {result.p_value:.4f}', 
             fontsize=13, fontweight='bold')
ax.set_xlabel('Time (Days)')
ax.set_ylabel('Survival Probability')
ax.legend()
plt.tight_layout()
plt.savefig('km_RBM10.png', dpi=150)
plt.show()

print(f'RBM10 mutation rate: {len(mutated)}/{len(merged)} patients ({len(mutated)/len(merged)*100:.1f}%)')
print(f'Log-rank p-value: {result.p_value:.4f}')

## 7. Summary & Limitations

**Findings:**
- 194,731 somatic mutations were identified across 575 LUAD patients
- Among 8 established driver genes, RBM10 showed a statistically significant association with worse overall survival (log-rank p=0.0228)
- STK11 trended toward significance (p=0.0842) but did not cross the threshold
- Well-known drivers TP53, KRAS, and EGFR did not show significant survival differences in this cohort alone

**Limitations:**
- The RBM10-mutated group is small (n=44), resulting in wide confidence intervals and reduced statistical power
- No multivariate adjustment was performed — age, tumor stage, and smoking history could confound the RBM10 survival association
- This is an observational analysis; causality cannot be inferred

**Next steps:**
- Cox proportional hazards regression to control for age and AJCC stage
- Expand gene list to all genes mutated in >30 patients
- Validate RBM10 finding in an independent LUAD cohort (e.g. CPTAC-LUAD)